# Objective
Our objective is to capture and analyse in real-time telemetry emitted by a series of events generated by transactions passing through non-linear sequences of noisy processes with individually varying processing latency and service rates.  

Transactions are created at the root of processing trees, and generate events in subsequent processes in that same tree as they pass through.

In this notebook we simulate those telemetry data and explore approaches to inferring the processing tree and identifying anomalies.

## Our Approach vs Current Marketplace 
The *first critical dependency* to being able to make use of event data is to be able to faithfully determine the processinging tree through which a transaction passed.  Current offerings require one or more of the following to achieve this:
- publishers to have knowledge of the processes upstream and/or downstream of them (or indeed the entire tree), and to publish this information with the event data or to configure it centrally within the collection process,
- publishers to use a common identifier accross all events generated by a transaction,
- more advanced approaches automatically inject UUIDs into message headers and rely on processes to pass these on to subsequent events in the tree.  This can be problematic when transactions run accross different technologies which in turn requires custom logic to maintain these UUIDs across the chain.

Here we take a different approach by attempting to abstract away the need for processes to know anything about other processes in the tree.  Our requirements are minimal:
- a process must generate an event at a consistent point in its processing logic (it doesn't matter where, just that it is consistent within that process)
- the event must be uniquely identifiable via an event name (and optionally other identifiers)
- the event must contain a transaction reference that is also published by at least one other event in the tree.  Events can contain multiple transaction references, and not all processes need use a common ID.

The *second critical dependency* to being able to make use of event data is to preserve the granularity of the original observations.  The quantity of telemetry generated by a process is at least a factor if not several orders of magnitude in volume greater than its transaction data.  Think of a trasnaction that passes through 10 processes: in order to instrument each of those processes, 10 events will be generate for each transaction.  The general approach in the current marketplace is to downsample telemetry data however this approach precludes subsequent application of the vast majority of statistical techniques (such as t-tests, linear correlation, and fitting data to probability distributions) and limiting analysis to the summary statistics at the granularity at which data is now retained. 

Our approach is contrary to trend in that we attempt to store original observations in such a way that supports efficient retrieval.

Our assumption is that by centralising this logic within the collecting processes and raising the level of abstraction for publishers and consumers, the overall cost of collecting and correlating event data is optimised where there are many publishers and a single collector.  This does of course require complex inference at scale on the part of the collector, and the exploration of those problems and possible solutions is what this notebook sets out to prove as feasible.


## Our problem statements - the many unkowns
In the system described above, the collector is tasked with correlating events and will face the following challenges:

1. The rate at which transactions are created is stochastic, highly variable and the distribution is unknown in advance.  However, under the Central Limit Theorum we can assume that the means of samples from that distribution are normally distributed.  TODO: check this assumption with Dimitrios, Ola or a Mathematician.

2. The time taken for a transaction to pass through a given process is stochastic, similarly assumed to follow a normal distribution but the parameters of that distribution are unknown in advance.

3. No assumptions can be made as to the sequence in which events are observed by our telemetry process: events from multiple differing event chains may be interleaved, events may be received out of order etc.

4. The number of nodes in the processing tree nor their adjacency is known in advance.  However it is possible to identify the exact process in which an event was generated and associated timestamps may provide clues as to the edges between nodes.

5. The correlation of events into a causally related event chain/tree will only be possible deterministically via common transaction identifiers.  Each event will reference one or more identifiers with the sole constraint that each event must share an identifier with at least one other event in the chain.  There can be any number of transaction identifiers in an event chain.  This means that individual events in the chain may have mutually exclusive identifiers, and events and identifiers can be receieved in any order.

6. The rate at which transactions are created, at times, can exceed the service rates of processes in the tree, such that a queue may form at the ingress of those processes.  Similarly processes in the tree may have a service rate lower than proceeding processes similarly resulting in queuing above a certain throughput rate.

7. Exceptionally, processes may exhibit anomalous behaviour, orthogonal to queing.  This includes: a) anomalous processing latency, b) anomalous service rate, c) inaccurate data published to the telemetry collector.

8. The breakdown of latency between events due to processing time, queuing, anomalous behaviour cannot be directly observed as no direct information regarding state is passed in the event message.

...phew.  Reader - if these seem like genuine and motivating challenges, please read on...


## Modelling & Simplifying Assumptions in this notebook
The *originating process*, T, will be modelled as emitting n events per time unit, where the mean per time unit follows a lognormal distribution.  The timestamps of those events per time unit will be entirely random, but rate-limited to a fixed service rate, α, events per second.

Each subsequent *process*, A, B, C etc. will be modelled as having 
- a processing latency following a normal distribution with standard parameters (μ, σ)
- a service rate of α tranactions per time unit
- a FIFO queue is assumed at the ingress of all processes with infinite possible queue depth
- while in reality processes are multi-threaded, this is not modelled explicitly, instead described by the processing latency and service rate

Each *event* will contain the following data points
- An event name/ID allowing the process generating the event to be uniquely identified
- One or more Transaction References unique to the event chain that must be shared with at least one other event in the chain
- The Timestamp at which the transaction was processed, generating the event


## Outcomes
This notebook will set out to assert:
1. It is possible to determine event chains via their related refences with O(1) complexity.  In other words, as we accumlate events and event chains, the time required to search for related references and build event chains does not increase.
2. Once individual event chains have been established, it is possible to determine the adjacency matrix of the processing tree by comparing the co-linearity of timestamps between processes, and by extension identify the originating and terminal nodes in the tree.
3. It is possible to determine both when queues have formed at the ingress of a process due to the arrival rate exceeding the service rate for that process
4. Once queuing (load-related latency) has been accounted for, it is possible to determine the usual processing latency for each process
5. It is possible to detect a state change in any given process due to queuing or otherwise
5. The desired outcomes are to detect:
- when queues are increasing and at which processes
- when the latency of a process changes (i.e. starts to follow a different distribution)
- the expected end-to-end processing time through the tree.  And by extension, any transactions that have yet to complete within this timeframe, or did complete processing through the tree but outside of the expected time range.


## Requirements for running this notebook
- Python 3.11 or above & ability to install packages from pypi
- Redis Stack 7.4.2 or above


## Terminology & Notation
- *Telemetry* - the remote measurement of a process, normally via second-order characteristics
- *Non-linear* - i.e. not in a strict singular sequence, a tree-like or network structure
- *Noisy* - not strictly predictable, non-deterministic
- *Processing latency* - the time taken for a transaction to pass through a process from the perspective of the process, i.e. not taking into account any queuing time before processing starts
- *Service rate* - the maximum throughput of a process
- *Transaction* - an entity that passess through a set of processes, denoted by capital T
- *Process* - a node in a graph of many processes, denoted by a capital letter, A, B, C, D, E
- *Event* - generated when a transaction passes through that particular process.  T1 would generate events A1, B1, C1 as it passes through processes A, B and C
- *Event Group* = events belonging to the same transaction: A1, B1, C1
- *Event Chain* = the sequence of events of a transaction: A1 -> B1 -> C1
- *Event Set* = events belonging to the same process accross all transactions: An, Bn, Cn
- *Event Tree* = the generalised superset of all event chains combined

In [ ]:
from datetime import datetime
import json
from io import StringIO
from abc import ABC, abstractmethod
from scipy import stats
from sklearn.tree import DecisionTreeClassifier as SklearnDT
import numpy as np
import random
import pandas as pd
import polars as pl
from plotly import express as px
from plotly import graph_objects as go
import ipycytoscape
import itertools
from hmmlearn import hmm
import redis
from ulid import ULID #nb python-ulid

# To run this notebook you will also need a locally running instance of Redis on default ports

## 1. Simulate Telemetry

In [2]:
"""
Set the parameters of the simulation

*Processing Tree*
A -> B -> D
A -> C -> E -> F
A -> C -> E -> G
A -> H  (for 50% of events only)

*Latencies and Service Rates per second*
T: μ = 5,   σ = 8,   α = 28
A: μ = 2,   σ = 0.8, α = 25
B: μ = 0.9, σ = 0.3, α = 25 (initally, moving to μ = 2, σ = 2, α = 25)
C: μ = 0.9, σ = 0.3, α = 25
H: μ = 1.5, σ = 0.6, α = 12.5
D: μ = 2.5, σ = 1,   α = 28
E: μ = 2.5, σ = 1,   α = 28
F: μ = 0.9, σ = 0.2, α = 7
G: μ = 0.5, σ = 0.1, α = 12.5

*Transaction IDs*
A: An
B: An, Bn
C: Cn, An
D: Dn, Bn
E: En, Cn
F: Fn, Cn, En
G: Gn, En + one of An or Cn,
H: An

*Start/Duration*
19th March 2025 at midday
120 seconds / time intervals
"""

START_TIME = np.datetime64("2025-03-19T12:00:00.000")
NUM_INTERVALS = 10
T_MU = int(np.log(15)) #1.5
T_SIGMA = int(np.log(50)) #1.7
T_ALPHA = 75
proc_params = {"A": {"mu":0.5, "sigma":0.5, "alpha":60, "dep":"T", "refs":["A"], "context":["tea"]},
               "B": {"mu":0.9, "sigma":0.3, "alpha":25, "dep":"A", "refs":["A", "B"]},
               "C": {"mu":0.9, "sigma":0.3, "alpha":25, "dep":"A", "refs":["C", "A"], "context":["tea", "coffee"]},
               "D": {"mu":2.5, "sigma":1.0, "alpha":25, "dep":"B", "refs":["D", "B"]},
               "E": {"mu":2.5, "sigma":1.0, "alpha":12, "dep":"C", "refs":["E", "C"]},
               "F": {"mu":0.9, "sigma":0.2, "alpha":60, "dep":"E", "refs":["F", "C", "E"], "context":["milkshake"]},
               "G": {"mu":0.5, "sigma":0.1, "alpha":7, "dep":"E", "refs":["G", "E", ["A", "C"]]},
               "H": {"mu":1.5, "sigma":0.6, "alpha":25, "dep":"A", "refs":["A"]},
}

In [3]:
def apply_service_rate(timeseries: np.ndarray, rate_per_second: int) -> np.ndarray:
    """ Function to apply a service rate (maximum throughput rate to a timeseries.

        Inputs:
        timeseries - and numpy array of type np.datetime64
        rate_per_second = a scalar integer indicating the number of events per second

        Returns:
        The original timeseries now throttled by the service rate, 
        as a numpy array of type np.datetime64

        Steps:
        1) Convert the timeseries to Pandas in order to make it easier to record the original order
        in a column named 'index'
        2) Caclulate the time delta between events, increasing these to the rate limit interval where they are smaller
        3) Calculate the new time series based on the rate-limited cumulative time deltas and return timestamps
        in the original order (which may not have been sorted by timestamp)
    """
    timeseries = pd.DataFrame(timeseries, columns=["timeseries"]).sort_values("timeseries")
    timeseries["time_deltas_between_events"] = timeseries["timeseries"].diff()
    timeseries["rate_limited_time_deltas_between_events"] = np.maximum(timeseries["time_deltas_between_events"], 
                                                            np.full(timeseries["time_deltas_between_events"].shape, np.timedelta64(int(1/rate_per_second*1000),"ms")))
    timeseries.loc[timeseries["rate_limited_time_deltas_between_events"].isna(), "rate_limited_time_deltas_between_events"] = np.timedelta64(0, "ms")
    timeseries["rate_limited_timeseries"] = timeseries["rate_limited_time_deltas_between_events"].cumsum() + timeseries.loc[0,"timeseries"]
    return timeseries.sort_index()["rate_limited_timeseries"].to_numpy()

In [4]:
""" Simulate the originating event
"""
T = np.ndarray([0], dtype=np.datetime64)
for i in range(NUM_INTERVALS):
    #First determine how many originating events we will generate for this time interval
    #T_num_emissions_this_interval = min(int(np.random.lognormal(mean=np.log(T_MU), sigma=np.log(T_SIGMA))), T_ALPHA)
    T_num_emissions_this_interval = min(int(stats.lognorm(s=T_SIGMA, scale=np.exp(T_MU)).rvs(1)[0]), T_ALPHA)

    if T_num_emissions_this_interval > 0:
        #Generate the number of events from above at entirely random timestamps within this time interval & append to a list
        T_this_Interval = np.sort(np.random.random(T_num_emissions_this_interval).round(3)*1000).astype(np.timedelta64) + START_TIME + np.timedelta64(i, "s")
        T = np.concatenate((T, T_this_Interval), axis=0)

NUM_OBSERVATIONS = T.shape[0]
print(f"Overall number of originating transactions: {NUM_OBSERVATIONS}")

Overall number of originating transactions: 352


In [5]:
""" Simulate processing time in all processes, subject to their service rate
"""
timestamps = {}
timestamps["T"] = T
for proc, params in proc_params.items():
    processing_latency_this_process = (np.absolute(stats.norm(loc=params["mu"], scale=params["sigma"]).rvs(NUM_OBSERVATIONS).round(3))*1000).astype(np.timedelta64)
    timestamps[proc] = apply_service_rate(timestamps[params["dep"]], params["alpha"]) + processing_latency_this_process
del timestamps["T"]
NUM_EVENTS = len(timestamps) * len(timestamps["A"])
print(f"Overall number of events: {NUM_EVENTS}")

Overall number of events: 2816


In [6]:
""" Simulate transaction references
"""
trans_refs = {}
for proc, params in proc_params.items():
    refs_this_process = []
    for i in range(NUM_OBSERVATIONS):
        refs_this_event = []
        refs = params["refs"]
        for ref in refs:
            if type(ref) ==  str:
                refs_this_event.append({"type": ref, "id": ref + str(i), "ver": 1})
            if type(ref) == list:
                ref = np.random.choice(ref, 1, p=[1/len(ref)]*len(ref))
                refs_this_event.append({"type": str(ref[0]), "id": ref[0] + str(i), "ver": 1})
        refs_this_process.append(refs_this_event)
    trans_refs[proc] = refs_this_process

In [7]:
""" Simulate additional context information
"""
context = {}
for proc, params in proc_params.items():
    context_items_this_process = []
    for i in range(NUM_OBSERVATIONS):
        context_items_this_event = {}
        try:
            context_items = params["context"]
            for item in context_items:
                context_items_this_event[item]=i
        except:
            pass
        #if len(context_items_this_event) > 0:
        context_items_this_process.append(context_items_this_event)
    context[proc] = context_items_this_process

In [8]:
""" Load events into a Pandas DataFrame to facilitate ordering roughly in timestamp order
    with some local re-shuffling
    and export as json
"""
event_data = pd.DataFrame([{"EventName":k, "Timestamp":val, "Refs":trans_refs[k][i],"Context":context[k][i]} for k,v in timestamps.items() for i, val in enumerate(v)])
event_data = event_data.sort_values("Timestamp").reset_index()
event_data["index"] = event_data["index"] + np.random.randint(0,NUM_OBSERVATIONS, event_data.shape[0])
event_data.set_index("index").sort_index().reset_index(drop=True).to_json("eventdata.json", orient="records")
event_data.shape, event_data.set_index("index").sort_index().reset_index(drop=True).sample(5)

C:\Users\Ben\AppData\Local\Temp\ipykernel_5608\1262874362.py:8: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  event_data.set_index("index").sort_index().reset_index(drop=True).to_json("eventdata.json", orient="records")


((2816, 5),
      EventName               Timestamp  \
 925          C 2025-03-19 12:00:06.235   
 847          C 2025-03-19 12:00:18.544   
 1723         E 2025-03-19 12:00:32.879   
 687          B 2025-03-19 12:00:16.400   
 1192         D 2025-03-19 12:00:16.310   
 
                                                    Refs  \
 925   [{'type': 'C', 'id': 'C86', 'ver': 1}, {'type'...   
 847   [{'type': 'C', 'id': 'C306', 'ver': 1}, {'type...   
 1723  [{'type': 'E', 'id': 'E288', 'ver': 1}, {'type...   
 687   [{'type': 'A', 'id': 'A257', 'ver': 1}, {'type...   
 1192  [{'type': 'D', 'id': 'D175', 'ver': 1}, {'type...   
 
                           Context  
 925     {'tea': 86, 'coffee': 86}  
 847   {'tea': 306, 'coffee': 306}  
 1723                           {}  
 687                            {}  
 1192                           {}  )

## 2. Group events by common reference IDs

#### A quick Redis/JSON Cribsheet
URLs:
- Redis Default Connection URL: http://localhost:6379
- Redis Insight: http://localhost:8001/redis-stack/browser

Redis Commands:
- FLUSHDB - delete the full contents of Redis including indices

Useful python functions for redis:
Converting to/from unix timestamp
- datetime.fromtimestamp(1610668800)
- datetime(2021,1,5, 0, 0).timestamp()

JSONPath cribsheet:
- $ - the root object or element
- @ - current object or element
- . - child operator, used to denote a child element of the current element
- .. - recursive scan
- \* - wildcard, returning all objects or elements regardless of their names
- [] - subscript operator / array operator
- , - union operator, returns the union of the children or indexes indicated
- : - array slice operator; you can slice arrays using the syntax [start:step]
- () - lets you pass a script expression in the underlying implementation’s script language
- ?()	- applies a filter/script expression to query all items that meet certain criteria

In [9]:
 #Load our simulated events into our application
with open("eventdata.json") as f:
    events = json.load(f)
print(f"{len(events)} events loaded")

#Create an empty adjacency matrix - here it will be empty, 
#but later we will populate, and we can see the effect on the search
adjacency_matrix = pd.DataFrame()

#Load the expected, source and terminal events
#This can be re-run a 2nd time after the adjacency matrix is created later in this notebook in order to see the effect
if len(adjacency_matrix) > 0:
    expected_events = set(adjacency_matrix["source"]) | set(adjacency_matrix["target"])
    source_nodex = set(adjacency_matrix["source"]) - set(adjacency_matrix["target"])
    terminal_nodes = set(adjacency_matrix["target"]) - set(adjacency_matrix["source"]) 
else:
    expected_events = set()
    source_nodex = set()
    terminal_nodes = set()

pool = redis.ConnectionPool(host="localhost", port=6379, decode_responses=True)
r = redis.Redis(connection_pool=pool)

2816 events loaded


In [11]:
REDIS_KEY_BASE = "argus:ec"
INDEX_NAME = "argus:ec:idx"
STREAM_NAME = "argus:ecstream"

def create_key():
    key = str(ULID())
    return f"{REDIS_KEY_BASE}:{key}"

def add_or_merge_event_to_event_chain(r, event):
    """ This function will search on the given redis connection for an existing event chain
        with any of the references in the given event.  It will create a new event chain or update
        existing event chains accordingly.  Finally it will push the created/update event chain id to a stream
    """

    #Workaround - we need to find event chains with tuples of (reftype, ref, id)
    #Here for ease we concatenate those three values to a string separated with underscore characters
    #TODO - look at alternate methods
    #Further it could be that an event itself contains multiple refs of the same type
    #TODO - in that case it would be necessary to split/duplicate the event and treat as 2 events
    #A bloom filter might be more performant in order to determine if any of the IDs have been seen before
    #TODO - experiment with a Bloomfilter - this exists in Redis Stack, and Python implementation of a 
    #Bloomfilter is below.  We also have a BF coded in Java already in our Next Gen DB
    event_concatenatedrefs = ["_".join([str(i) for i in ref.values()]) for ref in event["Refs"]]
    query_string = "@concatenatedrefs:{" + "|".join(event_concatenatedrefs) + "} "   
    query_string += "@complete:{false}"
    results = r.ft("argus:ec:idx").search(query_string)
    
    #TODO Multi-thread at this point based on chain ID modulo the number of threads/processes
    event_timestamp = {event["EventName"]:event["Timestamp"]}
    context = event["Context"]

    if len(results.docs) == 0:
        #create a new event chain
        j = {"concatenatedrefs": event_concatenatedrefs,
             "timestamps":event_timestamp,
             "context": context,
             "complete":False}                    
        chain_id = create_key()
        p = r.pipeline()
        p.json().set(chain_id, "$", j)
        p.xadd(STREAM_NAME, {"ecid": chain_id})
        p.execute()
        pass

    #TODO The code below is quite repetative - re-org/re-factoring opportunity!   
    else:
        p = r.pipeline()
        #if the references on the results are mutually exclusive, then merge those to a single document
        if len(results.docs) > 1:
            for i in range(len(results.docs)):
                for j in range(i+1, results.docs):
                    #TO DO
                    pass

        for result in results.docs:
            #store the id for this chain - we'll need to update it
            #get a full list of refs already on the chain
            #and workout those that are missing
            #finally get the process name and timestamp of this event
            chain_json_this_result = json.load(StringIO(result.json))
            refs_this_result = chain_json_this_result["concatenatedrefs"]
            events_this_result = chain_json_this_result["timestamps"].keys()
            chain_id_this_result = result.id
            refs_not_found = set(event_concatenatedrefs) - set(refs_this_result)

            if len(refs_not_found) == 0:
                #if all of the refs are already present, then it is just matter of adding this event to the chain
                #p.json().arrappend(chain_id_this_result, "$.timestamps", event_timestamp)
                p.json().merge(chain_id_this_result, "$.timestamps", event_timestamp)
                p.json().merge(chain_id_this_result, "$.context", context)


            elif len(refs_not_found) > 0:
                #work out if there is any potential conflict - we only want one ID type per chain
                idtypes_in_conflict = {ref.split("_")[0] for ref in refs_this_result} & {ref.split("_")[0] for ref in refs_not_found}                

                #if no conflicts then we can simply add this event & missing refs to the chain
                if len(idtypes_in_conflict) == 0:
                    #p.json().arrappend(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.context", context)
                    for ref in refs_not_found:
                        p.json().arrappend(chain_id_this_result, "$.concatenatedrefs", ref)

                #if conflicts: add the event to the chain
                #add the refs not in conflict to the chain
                #duplicate the chain
                #add the refs in conflict separately and individually to the chains
                else:
                    refs_not_in_conflict = [str(r) for r in refs_not_found if r.split("_")[0] not in idtypes_in_conflict]
                    refs_in_conflict = [str(r) for r in refs_not_found if r.split("_")[0] in idtypes_in_conflict]
                    refs_minus_conflicts = [r for r in refs_this_result if r.split("_")[0] not in idtypes_in_conflict]
                    #add to the existing chain
                    #p.json().arrappend(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.context", context)
                    for r in refs_not_in_conflict:
                        p.json().arrappend(chain_id_this_result, "$.concatenatedrefs", r)
                    #create a 2nd chain
                    chain_json_this_result["timestamps"].append(event_timestamp)
                    chain_json_this_result["concatenatedrefs"] = refs_minus_conflicts + refs_in_conflict
                    p.json().set(create_key(), "$", chain_json_this_result)           

            awaited_events = expected_events - set(events_this_result) - set(event_timestamp.keys())
            if set(event_timestamp.keys()) in expected_events and len(awaited_events) == 0:
                p.json().toggle(chain_id_this_result, "$.complete")
            p.xadd(STREAM_NAME, {"ecid": chain_id_this_result}, maxlen=1000000, approximate=True)

        p.execute()
    
for event in events:
    add_or_merge_event_to_event_chain(r, event)

In [12]:
#Read the event chain from the stream for processing / monitoring / alerting
#The most recent x amount of time can be read from the stream 
lookbackperiod_ms = 5 * 60 * 1000   #i.e. 5 mins
start_timestamp = int(datetime.now().timestamp()) * 1000 - lookbackperiod_ms
results = r.xrange(name=STREAM_NAME, min=str(start_timestamp)+"-0", max="+")
event_chain_keys = [r[1]["ecid"] for r in results]
if len(event_chain_keys) > 0:
    event_chains = r.json().mget(event_chain_keys, "$")
event_chains = [i[0] for i in event_chains]
print(f"fetched {len(event_chain_keys)} event chains")

#view latencies from the stream including the combined context etc.
from_event = "A"
to_event = "B"
pd.DataFrame(pd.DataFrame(event_chains)["timestamps"].apply(lambda x: datetime.fromtimestamp((x[to_event] - x[from_event])/1000)).dt.time.rename(f"Latency from {from_event} to {to_event}")).merge(
            pd.DataFrame(pd.DataFrame(event_chains)["context"].to_list()),\
            how="left", left_index=True, right_index=True)

fetched 2816 event chains


,Latency from A to B,tea,coffee,milkshake
0,00:00:01.583000,6,6,6
1,00:00:01.227000,3,3,3
2,00:00:01.102000,33,33,33
3,00:00:01.897000,10,10,10
4,00:00:02.208000,67,67,67
...,...,...,...,...
2811,00:00:07.008000,348,348,348
2812,00:00:07.693000,343,343,343
2813,00:00:07.370000,342,342,342
2814,00:00:06.352000,323,323,323


## 3. Infer the Adjacency Matrix

In [13]:
""" Step #3 Populate the Adjacency Matrix
"""
#first get the keys of the event chains we'd like to analyse
#here we simply get all of our toy data, but equally there could be an
#index on timestamp, or some other indexed attribute we could use to retrieve
#a sample population of event chains
keys = r.keys("argus:ec:*")
eventchain_timestamps = r.json().mget(keys, "$.timestamps")

#load into a DataFrame, sort by the column that has the lowest value
eventchain_timestamps = pd.DataFrame([d[0] for d in eventchain_timestamps])
minvals = eventchain_timestamps.min()
min_col = minvals.index[minvals.argmin()]
eventchain_timestamps = eventchain_timestamps.sort_values(min_col).reset_index(drop=True)

#Visualise the time series
fig = px.line(eventchain_timestamps.reset_index().melt("index"), x="index", y="value", color="variable")
fig.show()

In [ ]:
class AbsCorrelationEstimator(ABC):
    @abstractmethod
    def fit(self, df: pd.DataFrame, max_pval: float) -> np.ndarray:
        pass


class PearsonCorrelationEstimator(AbsCorrelationEstimator):
    def fit(self, df: pd.DataFrame, max_pval: float) -> np.ndarray:
        event_labels = df.columns.tolist()
        n = len(event_labels)
        event_sets = df.T.to_numpy()
        corr = np.zeros((n, n))

        for e in range(n):
            for d in range(n):
                if e == d:
                    continue
                index = ~np.isnan(event_sets[e]) & ~np.isnan(event_sets[d])
                if not index.any():
                    continue
                if not (event_sets[e][index] >= event_sets[d][index]).all():
                    continue
                r = stats.pearsonr(event_sets[e][index], event_sets[d][index], alternative="greater")
                if r[1] < max_pval:
                    corr[e, d] = r[0]

        return corr


class AdjacencyMatrixEstimator:
    def __init__(self, estimator: AbsCorrelationEstimator):
        self.estimator = estimator

    def fit(self, df: pd.DataFrame, max_pval: float) -> tuple[list, list, pd.DataFrame]:
        event_labels = df.columns.tolist()
        event_sets = df.T.to_numpy()
        corr = self.estimator.fit(df, max_pval)

        nodes = []
        edges = []

        for e in range(len(event_labels)):
            nodes.append({"data": {"id": event_labels[e], "label": event_labels[e]}})

            if (corr[e] == 0).all():
                print(f"Event {event_labels[e]} is a root event")
            else:
                d = np.argmax(corr[e])
                index = ~np.isnan(event_sets[e]) & ~np.isnan(event_sets[d])
                delta = event_sets[e][index] - event_sets[d][index]
                mean_delta = np.round(np.mean(delta), 3)
                std_delta = np.round(np.std(delta), 3)
                max_delta = np.round(np.max(delta), 3)
                min_delta = np.round(np.min(delta))
                label = "m={}\n s={}\n max={}\n min={}".format(mean_delta, std_delta, max_delta, min_delta)
                print(f"Event {event_labels[e]} is likely to be dependent on {event_labels[d]}", label)
                edges.append({"data": {"source": event_labels[d], "target": event_labels[e], "label": label, "isdirected": True}})

        adjacency_matrix = pd.DataFrame(
            [(e["data"]["source"], e["data"]["target"]) for e in edges],
            columns=["source", "target"]
        )
        return nodes, edges, adjacency_matrix

In [61]:
max_pval = 0.05

estimator = AdjacencyMatrixEstimator(PearsonCorrelationEstimator())
nodes, edges, adjacency_matrix = estimator.fit(eventchain_timestamps, max_pval)


Event A is a root event
Event B is likely to be dependent on A m=5764.316
 s=3623.799
 max=12253
 min=484
Event C is likely to be dependent on A m=5790.043
 s=3615.935
 max=12543
 min=543
Event D is likely to be dependent on B m=6259.151
 s=2892.507
 max=13194
 min=374
Event E is likely to be dependent on C m=13152.143
 s=6965.273
 max=25881
 min=1012
Event F is likely to be dependent on E m=2582.516
 s=1383.948
 max=5008
 min=649
Event G is likely to be dependent on E m=18221.859
 s=11314.803
 max=37930
 min=713
Event H is likely to be dependent on A m=6347.688
 s=3681.992
 max=13579
 min=552


In [67]:
#Visualise the inferred dependency tree as a directed acylic graph
elements = {"nodes": nodes, 
            "edges": edges}

cytoscapeobj = ipycytoscape.CytoscapeWidget()
cytoscapeobj.graph.add_graph_from_json(elements, directed=True)
cytoscapeobj.set_layout(name="dagre")   #dagre, klay
cytoscapeobj.set_style([{
                          "selector":"node",
                          "style":{
                             "label-position": "right",
                             "font-family": "arial",
                             "font-size": "12px",
                             "label": "data(label)"
                          }
                       },
                       {
                          "selector":"edge",
                          "style":{
                             "width": 4,
                             "line-color": "#888",
                             "target-arrow-color": "#888",
                             "target-arrow-shape": "triangle",
                             "curve-style": "bezier",
                             "label": "data(label)",
                             "text-wrap": "wrap",
                             "font-family": "arial",
                             "font-size": "8px",
                          }
                       },
                       
                    ])


cytoscapeobj

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'label-pos…

## Step 4 - Group event groups into common chains and learn to classify them

In [68]:
# Discover Event Profiles and classify chains

presence = eventchain_timestamps.notna()
signatures = presence.apply(lambda row: tuple(row), axis=1)
unique_sigs = signatures.unique()

profiles = []
chain_profile_labels = pd.Series(index=eventchain_timestamps.index, dtype=int)

# Build child map from adjacency edges
children = {}
for e in edges:
    src, tgt = e["data"]["source"], e["data"]["target"]
    children.setdefault(src, set()).add(tgt)

for pid, sig in enumerate(unique_sigs):
    mask = signatures == sig
    node_set = frozenset(col for col, present in zip(eventchain_timestamps.columns, sig) if present)

    # Terminal nodes: nodes whose children (per adjacency) are not in this profile's node_set
    terminal_nodes = frozenset(
        n for n in node_set
        if not (children.get(n, set()) & node_set)
    )

    count = mask.sum()
    profiles.append({
        "profile_id": pid,
        "node_set": node_set,
        "terminal_nodes": terminal_nodes,
        "chain_count": count,
        "fraction": count / len(eventchain_timestamps),
    })
    chain_profile_labels[mask] = pid

for p in profiles:
    print(f"Profile {p['profile_id']}: nodes={sorted(p['node_set'])}, "
          f"terminals={sorted(p['terminal_nodes'])}, "
          f"chains={p['chain_count']} ({p['fraction']:.1%})")


Profile 0: nodes=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'], terminals=['D', 'F', 'G', 'H'], chains=490 (100.0%)


In [69]:


class AbsClassificationMethod(ABC):
    @abstractmethod
    def fit(self, X: np.ndarray, y: np.ndarray) -> None: ...

    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray: ...

    @abstractmethod
    def feature_importances(self) -> list[tuple[str, float]]: ...


class DecisionTreeMethod(AbsClassificationMethod):
    def __init__(self, feature_names: list[str], max_depth: int = 5, min_samples_leaf: int = 10):
        from sklearn.tree import DecisionTreeClassifier
        self._tree = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        self._feature_names = feature_names

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        self._tree.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self._tree.predict(X)

    def feature_importances(self) -> list[tuple[str, float]]:
        return sorted(
            [(name, float(imp)) for name, imp in zip(self._feature_names, self._tree.feature_importances_) if imp > 0],
            key=lambda x: x[1], reverse=True,
        )


class ChainClassifier:
    def __init__(self, method: AbsClassificationMethod):
        self.method = method

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        self.method.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.method.predict(X)

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> float:
        preds = self.predict(X)
        return float((preds == y).sum()) / len(y)


# --- Build binary feature matrix (exclude terminal nodes) ---
all_terminals = frozenset().union(*(p["terminal_nodes"] for p in profiles))
feature_cols = [c for c in eventchain_timestamps.columns if c not in all_terminals]
X = eventchain_timestamps[feature_cols].notna().astype(np.int8).to_numpy()
y = chain_profile_labels.to_numpy()

# --- Fit and evaluate ---
method = DecisionTreeMethod(feature_names=feature_cols)
classifier = ChainClassifier(method)
classifier.fit(X, y)

accuracy = classifier.evaluate(X, y)
print(f"Accuracy: {accuracy:.1%}")
print("Feature importances:")
for name, imp in method.feature_importances():
    print(f"  {name}: {imp:.3f}")

Accuracy: 100.0%
Feature importances:


# Step 5 - Autocorrelate Latency with volume of events

In [71]:
source_event = "A"
target_event = "B"
bucket_seconds = 1  # time bucket granularity in seconds

# --- Filter to chains that have both events ---
pair = eventchain_timestamps[[source_event, target_event]].dropna()
pair = pair.rename(columns={source_event: "source_ts", target_event: "target_ts"})
pair["latency_ms"] = pair["target_ts"] - pair["source_ts"]

# --- Bucket by source event timestamp ---
t_min = pair["source_ts"].min()
pair["bucket"] = ((pair["source_ts"] - t_min) // (bucket_seconds * 1000)).astype(int)

bucketed = pair.groupby("bucket").agg(
    volume=("latency_ms", "count"),
    mean_latency=("latency_ms", "mean"),
).reset_index()
bucketed["time_offset_s"] = bucketed["bucket"] * bucket_seconds

# --- Plot dual-axis: volume bars + latency line ---
fig = go.Figure()
fig.add_trace(go.Bar(
    x=bucketed["time_offset_s"], y=bucketed["volume"],
    name="Volume", yaxis="y", opacity=0.6,
))
fig.add_trace(go.Scatter(
    x=bucketed["time_offset_s"], y=bucketed["mean_latency"],
    name="Mean Latency (ms)", yaxis="y2", mode="lines+markers",
))
fig.update_layout(
    title=f"Volume & Latency: {source_event} → {target_event} ({bucket_seconds}s buckets)",
    xaxis_title="Time offset (s)",
    yaxis=dict(title="Volume (chain count)", side="left"),
    yaxis2=dict(title="Mean Latency (ms)", side="right", overlaying="y"),
    legend=dict(x=0.01, y=0.99),
)
fig.show()

print(f"Chains with both events: {len(pair)}")
print(f"Buckets: {len(bucketed)}")


Chains with both events: 490
Buckets: 13


In [75]:
# Rolling cross-correlation on a time-sequence basis with volume & latency overlay

from sklearn.preprocessing import StandardScaler

volume = bucketed["volume"].to_numpy().astype(float)
latency = bucketed["mean_latency"].to_numpy().astype(float)
timestamps = bucketed["bucket"].to_numpy()

# Standardise both series (zero mean, unit variance)
scaler = StandardScaler()
vol_norm = scaler.fit_transform(volume.reshape(-1, 1)).ravel()
lat_norm = scaler.fit_transform(latency.reshape(-1, 1)).ravel()

# Rolling cross-correlation over a trailing window
n = len(vol_norm)
window = max(5, n // 5)
sig = 1.96 / np.sqrt(window)

rolling_corr = np.full(n, np.nan)
for i in range(window - 1, n):
    v = vol_norm[i - window + 1 : i + 1]
    l = lat_norm[i - window + 1 : i + 1]
    rolling_corr[i] = np.corrcoef(v, l)[0, 1]

significant = np.abs(rolling_corr) > sig

# Identify contiguous significant spans
spans = []
in_span = False
for i in range(n):
    if significant[i] and not in_span:
        span_start = i
        in_span = True
    elif not significant[i] and in_span:
        spans.append((span_start, i - 1))
        in_span = False
if in_span:
    spans.append((span_start, n - 1))

# For each significant span, compute the lag via local cross-correlation
span_details = []
for start, end in spans:
    v_seg = vol_norm[start : end + 1]
    l_seg = lat_norm[start : end + 1]
    seg_n = len(v_seg)
    if seg_n < 3:
        span_details.append((start, end, np.nanmean(rolling_corr[start : end + 1]), 0, 0.0))
        continue
    seg_xcorr = np.correlate(v_seg, l_seg, mode="full") / seg_n
    seg_lags = np.arange(-(seg_n - 1), seg_n)
    peak_idx = np.argmax(np.abs(seg_xcorr))
    best_lag = seg_lags[peak_idx]
    best_r = seg_xcorr[peak_idx]
    mean_r = np.nanmean(rolling_corr[start : end + 1])
    span_details.append((start, end, mean_r, best_lag, best_r))

# Plot with three y-axes: volume (bar), latency (line), correlation (line)
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=timestamps, y=volume, name="Volume",
           marker_color="rgba(100, 149, 237, 0.4)"),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=timestamps, y=latency, name="Mean latency (ms)",
               mode="lines", line=dict(color="orange", width=2)),
    secondary_y=True,
)

fig.add_trace(
    go.Scatter(x=timestamps, y=rolling_corr,
               name="Rolling r (window={})".format(window),
               mode="lines", line=dict(color="darkred", width=2),
               yaxis="y3"),
)

for start, end in spans:
    fig.add_vrect(
        x0=timestamps[start], x1=timestamps[end],
        fillcolor="rgba(255, 0, 0, 0.08)", line_width=0,
    )

fig.add_trace(
    go.Scatter(x=[timestamps[0], timestamps[-1]], y=[sig, sig],
               mode="lines", line=dict(color="grey", dash="dot", width=1),
               showlegend=False, yaxis="y3"),
)
fig.add_trace(
    go.Scatter(x=[timestamps[0], timestamps[-1]], y=[-sig, -sig],
               mode="lines", line=dict(color="grey", dash="dot", width=1),
               showlegend=False, yaxis="y3"),
)

fig.update_layout(
    title="Volume, Latency & Rolling Correlation ({} \u2192 {})".format(
        source_event, target_event),
    xaxis_title="Time",
    yaxis=dict(title="Volume (chains)", side="left"),
    yaxis2=dict(title="Mean latency (ms)", side="right", overlaying="y"),
    yaxis3=dict(
        title="Pearson r", side="right", overlaying="y",
        anchor="free", position=1.0,
        range=[-1, 1],
        showgrid=False,
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(r=100),
)
fig.show()

# Summary table of significant periods with lag analysis
if span_details:
    print("Significant correlation periods (|r| > {:.3f}):\n".format(sig))
    rows = []
    for start, end, mean_r, lag, peak_r in span_details:
        t0 = pd.Timestamp(timestamps[start])
        t1 = pd.Timestamp(timestamps[end])
        duration = t1 - t0 + pd.Timedelta(seconds=bucket_seconds)
        lag_seconds = lag * bucket_seconds
        if lag > 0:
            lead = "volume leads"
        elif lag < 0:
            lead = "latency leads"
        else:
            lead = "contemporaneous"
        rows.append({
            "From": str(t0),
            "To": str(t1),
            "Duration": str(duration),
            "Mean r": round(mean_r, 3),
            "Peak lag r": round(peak_r, 3),
            "Lag (buckets)": lag,
            "Lag (seconds)": lag_seconds,
            "Interpretation": lead,
        })
    lag_df = pd.DataFrame(rows)
    display(lag_df)
else:
    print("No periods of significant correlation detected "
          "(threshold=\u00b1{:.3f})".format(sig))


Significant correlation periods (|r| > 0.877):



,From,To,Duration,Mean r,Peak lag r,Lag (buckets),Lag (seconds),Interpretation
0,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000010,0 days 00:00:01.000000002,0.962,1.57,0,0,contemporaneous


# Step 6 - Detect State Changes

## Detecting State Changes in Partially Observable Processes

In this section we look at different methods for detecting changes in an application process by purely looking its outputs rather than its internal workings.

## Terminology
*1) Partially Observable Processes*

With complex systems it may be more practical (or indeed the only realistic option) to measure the output of a process, rather than the process itself.  For example it may be more practical to roll a dice many times to see if it is fair rather than to perform physical measurements to see if it is perfectly weighted.  Or for example it may be more practical to measure the processing time (latency) of a process, rather than fully profile the process itself (CPU, Memory, Queue Depth, Free Threads, Handels etc.) 

When we cannot (or do not) observe a process directly, but rather through its second-order characteristics we say that it is 'Partially Observable', i.e. we can infer the inner state of the process from those secondary characteristics, but do not observe the process directly itself.

Of course there is nothing preventing us from measuring both first and second-order characterstics, but understanding their differences and applying the correct treatment to those data is essential.

See the following podcast on this topic: https://dataskeptic.com/blog/episodes/2015/partially-observable-state-spaces

*2) Stochastic*
A stochastic process is one where we cannot say with certainty how it will behave, but where we can describe its behaviour using a probability distribution model.  For example we can say that the trade booking latency in TCW follows a normal distribution with the parameters of 2 seconds mean and with a standard deviation of 0.8 seconds, but we cannot predict individually the processing time of each and every trade.

*3) Markov Process*
The Markov assumption is normally applied in order to *simplify* modelling of a process.  The Markov assumption is that the process is memory-less, i.e. that it's future outcomes do not depend on its history.

## The Occasionally Unfair Casino problem
The Occasionally Unfair Casino problem is normally explained as follows:

You are at a Casino gambling on the fact a die will roll any number between 1 and 5, but not the number six.  Iniitally the die is fair, so the chance of rolling a 6 is 1-in-6.  And this is what you see, approximately 1-in-6 throws result in the number 6.

However, at some point the Casino swaps the fair die for a biased one where the chance of now rolling a 6 is 1-in-2.  Now what you see is many sucessive 6s.

The issue is this: it is possible that the fair die could also roll many sucessive 6s.  How do you know if the die is still fair?  And if not, what are the new odds?

If we assume that our process behaves in the same was as the Occasionally Unfair Casino, i.e.
- the state of the system can only be indirectly observed via a secondary characteristic
- the system is stochastic
- that only the current state has a bearing on future states

Then there are a number of models we can use to detect the point at which the process becomes 'unfair' including Hidden Markov Models.  We also include fixed thresholds and moving average techniques below for comparison.


In [76]:
# We create a class called StateEstimator that will allow us to fit a model to our training data
# as well as estimate the states of our test data using that same model.  StateEstimator
# allows for three different concrete strategies for fitting and estimating data:
# 1) fixed thresholds, 2) a comparison of the moving average of 2 time windows, 3) a Hidden Markov Model

# The models are trained requiring one single input (from a Human Being) - the number of states in the 
# training data

# Further, the Hidden Markov Model can optionally accept a 'tansition matrix', i.e. the probability of
# moving from one state to the next.

class AbsStateEstimator(ABC):
    """ Interface class for the different models available to the StateEstimator class
    """
    @abstractmethod
    def fit(self, observations):
        pass

    @abstractmethod
    def estimate(self, observations):
        pass

class StateEstimator:
    """ This is the main object for inferring the state from observations.
        This class will train a model on a prior set of observations via it's fit method.
        The model can then be used to estimate the state of the system from a new set of 
        observations.
    """
    def __init__(self, model: AbsStateEstimator):
        self._model = model
    
    def fit(self, observations, n_states):
        self._model.n_states = n_states
        return self._model.fit(observations)
    
    def estimate(self, observations):
        return self._model.estimate(observations)
    

class FixedThresholdEstimator(AbsStateEstimator):
    """ This is a concrete class for estimating state changes using fixed thresholds
    """
    def __init__(self):
        self.n_states = None
        self.thresholds = None

    def fit(self, observations):
        from sklearn.mixture import GaussianMixture
        model = GaussianMixture(n_components=self.n_states)
        obs = observations.reshape(-1, 1)
        model.fit_predict(obs)

        pdf_params = np.sort(np.stack((model.means_.flatten(), np.sqrt(model.covariances_.flatten())), axis=1), axis=0)

        thresholds = []
        x = np.arange(np.min(observations), np.max(observations), 0.1)
        for i in range(len(pdf_params) - 1):
            pdf = stats.norm.pdf(x, pdf_params[i][0], pdf_params[i][1])
            nextpdf = stats.norm.pdf(x, pdf_params[i+1][0], pdf_params[i+1][1])
            intersection_x = np.argwhere(np.diff(np.sign(pdf - nextpdf))).flatten()
            intersection_y = pdf[intersection_x]
            thresholds.append(intersection_y)
            
        self.thresholds = np.array(thresholds)

    def estimate(self, observations):
        estimated_states = np.zeros_like(observations)
        for state, threshold in enumerate(self.thresholds):
            index = np.where(observations >= threshold)
            estimated_states[index] = state
        return estimated_states


class MovingAverageEstimator(AbsStateEstimator):
    def __init__(self):
        self.window = 5
        self.offset = 5
        self.numberofstates = None
        self.observations = None
        self.threshold = 3

    def fit(self, observations):
        pass

    def estimate(self, observations):
        return self._estimate(observations, self.threshold)

    def _estimate(self, observations, threshold):
        estimated_states = np.zeros_like(observations)

        multiplier = np.repeat([1/self.window],self.window)

        ma = np.concatenate((np.zeros(self.window - 1), np.convolve(observations, multiplier, mode="valid")))
        offsetma = np.concatenate((np.zeros(self.offset), ma[:-self.offset]))

        estimated_states_dec = np.zeros_like(observations)

        index = ma / offsetma > threshold
        estimated_states[index] = 1
        index = offsetma > ma * threshold
        estimated_states[index] = -1
        estimated_states[:self.window + self.offset] = 0

        estimated_states = np.cumsum(np.trunc(np.convolve(estimated_states, multiplier, mode="valid")))

        estimated_states -= np.min(estimated_states)

        return estimated_states


class HMMGaussianStateEstimator(AbsStateEstimator):
    """ 
    """
    def __init__(self):
        self.NUMBER_TRIALS = 5
        self.NUMBER_ITER = 100
        self.numberofstates = 3
        self.transmat = None
        self.best_score = None
        self.best_model = None
    
    def fit(self, observations):

        obs = observations.reshape(-1, 1)

        if type(self.transmat) == np.ndarray:
            init_params = "msc"
        else:
            init_params = "mtsc"

        model = hmm.GaussianHMM(n_components=self.numberofstates, n_iter=self.NUMBER_ITER, \
                                tol=1e-3, covariance_type="full", init_params=init_params)

        for i in range(self.NUMBER_TRIALS):

            if type(self.transmat) == np.ndarray:
                model.transmat_ = self.transmat

            model.fit(obs)
            score = model.score(obs)

            if not self.best_score or self.best_score < score:
                self.best_score = score
                self.best_model = model

    def estimate(self, observations):

        obs = observations.reshape(-1, 1)

        return self.best_model.predict(obs)

    @property
    def modelparams(self):
        return {"means": np.round(self.best_model.means_,2),
                "stds": np.round(self.best_model.covars_ ** 0.5, 2),
                "transmat": np.round(self.best_model.transmat_, 3),
                "startprob": np.round(self.best_model.startprob_, 2),
        }

In [77]:
""" Generate toy data for training
    The following generates a random number of observations from the normal distributions
    whose mean and std are described in the dictionary states.
    We plot these on a single timeline, colour-coding each segment of the line according
    to its state to help us visualise our toy data
"""
NUM_TRANSITIONS = 7
MAX_OBS_PER_STATE = 100
states = [{"loc":2.4,
           "scale":0.9},
          {"loc":4.5, 
           "scale":5},
          {"loc":20, 
           "scale":10}
        ]
state_membership = []
observations = np.ndarray([])
for i in range(NUM_TRANSITIONS):
    random_state = int(np.floor(random.random()*3))
    num_observations = int(np.round(random.random()*MAX_OBS_PER_STATE))
    observations = np.append(observations, stats.norm(**states[random_state]).rvs(num_observations).round(3))
    index = np.where(observations < 0.01)
    observations[index] = 0.01
    state_membership.extend(np.repeat(random_state, num_observations))
    print("Added {0} observations from state {1}".format(num_observations, random_state))

graph_colours = {0:"red",
                 1:"orange",
                 2:"green"}
fig = go.Figure()

start = 0
for i, g in itertools.groupby(state_membership):
    length = len(list(g))
    end = start + length
    fig.add_trace(go.Scatter(x = list(range(start, end)),
                             y = observations[start:end],
                            mode = "lines",
                            line=(dict(color=graph_colours[i]))
                        )
                )

    print("plotted state {0} from {1} to {2}".format(i, start, end))
    start = end

fig.show()

Added 47 observations from state 0
Added 62 observations from state 2
Added 51 observations from state 2
Added 87 observations from state 1
Added 4 observations from state 0
Added 10 observations from state 2
Added 13 observations from state 0
plotted state 0 from 0 to 47
plotted state 2 from 47 to 160
plotted state 1 from 160 to 247
plotted state 0 from 247 to 251
plotted state 2 from 251 to 261
plotted state 0 from 261 to 274


In [78]:
# We now throw away the information about the generating process, and just keep the observed data.
# And attempt to determine the different states from the observed data.

# Uncomment/Comment the model you would like to use to try to infer the different states from the observed data

#model = FixedThresholdEstimator()
#model = MovingAverageEstimator()
model = HMMGaussianStateEstimator()
model.transmat = np.array([[0.998,0.001,0.001],[0.001,0.998, 0.001],[0.001,0.001,0.998]])

estimator = StateEstimator(model)
estimator.fit(observations, n_states=3)

estimated_states = model.estimate(observations)

graph_colours = {0:"red",
                 1:"orange",
                 2:"green",
                 3:"blue",
                 4:"purple"}
fig = go.Figure()

start = 0
for i, g in itertools.groupby(estimated_states):
    length = len(list(g))
    end = start + length
    fig.add_trace(go.Scatter(x = list(range(start, end)),
                             y = observations[start:end],
                            mode = "markers",
                            line=(dict(color=graph_colours[i]))
                        )
                )

    print("plotted state {0} from {1} to {2}".format(i, start, end))
    start = end

fig.show()

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'
Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'
Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Ev

plotted state 1 from 0 to 48
plotted state 0 from 48 to 161
plotted state 2 from 161 to 248
plotted state 1 from 248 to 252
plotted state 0 from 252 to 262
plotted state 1 from 262 to 275


### WIP FROM THIS POINT ONWARDS
### Queuing Crib Sheet

- λ = arrival rate
- α = service rate
- ρ = traffic intensity = λ / α
- W = average time a customer spends in the system
- L = averge number of customers in the system = λW
- Lq = average number of customers in the queue except those being served = ρ / (1-ρ)



In [ ]:
NUM_CHAINS = eventchain_timestamps.shape[0]
cum_observation_count = (np.arange(NUM_CHAINS) + 1)
alpha = pd.DataFrame(index=eventchain_timestamps.index)
lambd = pd.DataFrame(index=eventchain_timestamps.index)
rho = pd.DataFrame(index=eventchain_timestamps.index)
print(f"Number of chains: {NUM_CHAINS}")

In [ ]:
for col in eventchain_timestamps.columns:
    #Alpha average service rate per second for the whole period sampled.  TODO think about having a sampling window
    minval = eventchain_timestamps[col].min()
    alpha[col] = (cum_observation_count / (eventchain_timestamps[col].sort_values() - minval).T * 1000).sort_index()
    alpha = alpha.replace(np.inf, 0)
    #Lambda is the Alpha of the source
    adj = adjacency_matrix.loc[adjacency_matrix["target"]==col, "source"]
    if len(adj) > 0:
        source = adj.to_list()[0]
    else:
        source = col
    lambd[col] = alpha[source]
    rho = lambd / alpha
    Lq = rho / (1 - rho)

#alpha.to_csv("alpha.csv")
alpha